In [1]:
from datetime import datetime
import os
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import StringIO
import json
import logging


# 🔐 For testing ONLY - NEVER commit this to version control!
CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=thriveclientdatabase;AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;EndpointSuffix=core.windows.net"

CONTAINER_NAME = "thrive-client-ecosoulhome"

# 📅 Dynamic Path Construction
today = datetime.now()
BLOB_NAME = f"data_dump/inventory_ledger/amazon/{today.strftime('%Y')}/{today.strftime('%m')}/{today.strftime('%d')}/us_inventory_ledger.ndjson"
# BLOB_NAME = f"data_dump/inventory_ledger/amazon/2026/05/02/ca_amazon_inventory.ndjson"

print(f"📍 Blob Path: {BLOB_NAME}")

try:
    # Initialize Azure client
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=BLOB_NAME)
    
    # Download and parse
    download_stream = blob_client.download_blob()
    blob_text = download_stream.readall().decode('utf-8')
    
    # Handle empty file
    if not blob_text.strip():
        print("⚠️  File is empty")
        df = pd.DataFrame()
    else:
        df = pd.read_json(StringIO(blob_text), lines=True)
    
    print(f"✅ Loaded {len(df):,} records from {BLOB_NAME}")
    
except Exception as e:
    
    print(f"❌ Error: {type(e).__name__}: {e}")
    df = None

df

📍 Blob Path: data_dump/inventory_ledger/amazon/2026/06/16/us_inventory_ledger.ndjson
✅ Loaded 2,468 records from data_dump/inventory_ledger/amazon/2026/06/16/us_inventory_ledger.ndjson


,Date,FNSKU,ASIN,MSKU,Title,Disposition,Starting Warehouse Balance,In Transit Between Warehouses,Receipts,Customer Shipments,...,Warehouse Transfer In/Out,Found,Lost,Damaged,Disposed,Other Events,Ending Warehouse Balance,Unknown Events,Location,Store
0,2026-06-14,B08MXZJFLR,B08MXZJFLR,BCS350,ECO SOUL 100% Compostable Wooden Cutlery Set 3...,SELLABLE,70,0,0,-1,...,0,0,0,0,0,0,69,0,CA,
1,2026-06-14,B08MXZJFLR,B08MXZJFLR,BCS350,ECO SOUL 100% Compostable Wooden Cutlery Set 3...,WAREHOUSE_DAMAGED,1,0,0,0,...,0,0,0,0,0,0,1,0,CA,
2,2026-06-14,B08MY3HC8X,B08MY3HC8X,BCS175,ECO SOUL 100% Compostable Wooden Cutlery Set –...,SELLABLE,54,1,0,-4,...,1,0,0,0,0,0,51,0,CA,
3,2026-06-14,B08MYG6FW6,B08MYG6FW6,PLP10SQ100-CAN,"ECO SOUL Palm Leaf Like Bamboo Plates, 10 Inch...",CUSTOMER_DAMAGED,2,0,0,0,...,0,0,0,0,0,0,2,0,US,
4,2026-06-14,B08MYG6FW6,B08MYG6FW6,PLP10SQ100-CAN,"ECO SOUL Palm Leaf Like Bamboo Plates, 10 Inch...",SELLABLE,108,0,0,0,...,0,0,0,0,0,0,108,0,CA,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463,2026-06-06,X00508LOBH,B0GJ5K8PRY,PBFHW30P04,ECO SOUL Plant-Based Feminine Hygiene Wipes (3...,DEFECTIVE,3,0,0,0,...,0,0,0,0,0,-3,0,0,US,
2464,2026-06-06,X00508LOBH,B0GJ5K8PRY,PBFHW30P04,ECO SOUL Plant-Based Feminine Hygiene Wipes (3...,SELLABLE,44,120,0,-1,...,-10,0,0,0,0,3,36,0,US,
2465,2026-06-06,X00508PJ3V,B0GJ5LL6W3,PBSW30P01,ECO SOUL Plant-Based Sunscreen Wipes SPF 50 PA...,SELLABLE,112,448,0,0,...,0,0,0,0,0,0,112,0,US,
2466,2026-06-06,X00508PJ45,B0GJ59KYCH,PBFHW30P02,ECO SOUL Plant-Based Feminine Hygiene Wipes (3...,DEFECTIVE,95,0,0,0,...,0,0,0,0,0,-90,5,0,US,


In [2]:
import pandas as pd
from azure.storage.blob import BlobServiceClient
from io import StringIO
pattex_inventory =df.copy()
# -------   - STEP 1: CLEAN COLUMN NAMES --------


# Apply cleaning

# -------- STEP 2: AZURE CONNECTION --------
connection_string = "DefaultEndpointsProtocol=https;AccountName=thriveclientdatabase;AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;EndpointSuffix=core.windows.net"
container_name = "thrive-client-pattex"
blob_name = "sanity_data/temp.csv"   # e.g. folder/file.csv

# Create Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Get Blob Client
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

# -------- STEP 3: CONVERT DF TO CSV --------
csv_buffer = StringIO()
pattex_inventory.to_csv(csv_buffer, index=False)

# -------- STEP 4: UPLOAD TO AZURE --------
blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)

print("✅ DataFrame successfully uploaded to Azure Blob Storage!")

✅ DataFrame successfully uploaded to Azure Blob Storage!
